# Aula — Feature Store no Databricks

Versão preparada a partir do material da Alura.

Objetivos:
1. Verificar o catálogo e a tabela Iris.
2. Construir identificadores para as flores.
3. Separar as quatro features da variável `species`.
4. Registrar a Feature Table, se ainda não existir.
5. Preparar a característica `petal_ratio`.

Execute as células no Databricks Free Edition.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalogo", "workspace")

catalogo = dbutils.widgets.get("catalogo").strip()

import re
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo):
    raise ValueError("Catalogo invalido")

tabela_iris = f"{catalogo}.default.iris_dataset"
tabela_features = f"{catalogo}.default.iris_features_table"

print("ORIGEM:", tabela_iris)
print("DESTINO:", tabela_features)

if not spark.catalog.tableExists(tabela_iris):
    raise RuntimeError(
        "Tabela Iris ausente. Importe iris_dataset.csv "
        "no catalogo antes de continuar."
    )

df_iris = spark.table(tabela_iris)

colunas = [
    "sepal_length", "sepal_width",
    "petal_length", "petal_width", "species"
]

faltando = set(colunas) - set(df_iris.columns)

if faltando:
    raise RuntimeError(f"Colunas ausentes: {faltando}")

quantidade = df_iris.count()

if quantidade != 150:
    raise RuntimeError(
        f"Esperados 150 registros, encontrados {quantidade}"
    )

print("DATASET IRIS VALIDADO:", quantidade)
display(df_iris.limit(5))

## Criando a chave da Feature Store

A ordenação usa os dados das flores, evitando a ordenação
arbitrária do exemplo original.

Se houver linhas idênticas, seus IDs podem ser intercambiados
entre execuções. Não utilize esse método para identificar
entidades reais em produção.

In [ ]:
ordenacao = [
    F.col("sepal_length"),
    F.col("sepal_width"),
    F.col("petal_length"),
    F.col("petal_width"),
    F.col("species")
]

janela = Window.orderBy(*ordenacao)

df_identificado = df_iris.withColumn(
    "id_flor",
    F.concat(
        F.lit("flor_"),
        F.row_number().over(janela).cast("string")
    )
)

df_features = df_identificado.select(
    "id_flor",
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width"
)

if "species" in df_features.columns:
    raise RuntimeError("Target indevidamente incluido")

if df_features.select("id_flor").distinct().count() != 150:
    raise RuntimeError("Chaves duplicadas")

print("FEATURES PREPARADAS:", df_features.columns)
print("TARGET SPECIES: EXCLUIDO")
display(df_features.limit(10))

## Registrando no Unity Catalog

Esta etapa cria a tabela somente se ela não existir.

Se já houver uma Feature Table, o código a preserva.
O Databricks precisa permitir o uso do FeatureEngineeringClient.

In [ ]:
try:
    from databricks.feature_engineering import FeatureEngineeringClient
except ImportError as erro:
    raise RuntimeError(
        "Biblioteca databricks-feature-engineering ausente. "
        "Instale-a no notebook com %pip e reinicie o Python."
    ) from erro

fe = FeatureEngineeringClient()

if spark.catalog.tableExists(tabela_features):

    existente = spark.table(tabela_features)

    if existente.count() != 150:
        raise RuntimeError(
            "Feature Table existente possui contagem diferente"
        )

    obrigatorias = set(df_features.columns)

    if not obrigatorias.issubset(set(existente.columns)):
        raise RuntimeError(
            "Feature Table existente tem esquema incompatível"
        )

    print("FEATURE TABLE EXISTENTE PRESERVADA")

else:

    fe.create_table(
        name=tabela_features,
        primary_keys=["id_flor"],
        df=df_features,
        schema=df_features.schema,
        description="Features morfologicas do dataset Iris."
    )

    print("FEATURE TABLE CRIADA")

display(spark.table(tabela_features).limit(10))

## Preparando a feature petal_ratio

Calcula a razão entre comprimento e largura da pétala.

Esta etapa é somente de visualização: não atualiza nem
remove colunas da tabela original.

In [ ]:
df_atual = spark.table(tabela_features)

df_ratio = df_atual.select(
    "id_flor",
    F.when(
        F.col("petal_width") > 0,
        F.col("petal_length") / F.col("petal_width")
    ).alias("petal_ratio")
)

if df_ratio.filter(
    F.col("petal_ratio").isNull()
).limit(1).count():
    raise RuntimeError("Razao invalida ou divisao por zero")

print("NOVA FEATURE CALCULADA: petal_ratio")
print("TABELA ORIGINAL PRESERVADA")
display(df_ratio.limit(10))

## Próxima atividade

Estudar a atualização da Feature Table com `write_table`.

Antes dessa operação:
- Conferir se `petal_ratio` já existe.
- Validar a compatibilidade do esquema.
- Conferir as permissões de escrita.
- Evitar o `DROP COLUMNS` do notebook do professor.

Nenhuma alteração destrutiva foi incluída.

# Continuação — Criando e atualizando Features

Nesta etapa utilizaremos o FeatureEngineeringClient
criado nas células anteriores.

A nova feature `petal_ratio` será gravada por MERGE.

O notebook preserva a tabela existente e impede a
exclusão acidental da feature oficial.


In [ ]:
print("===== CONFERINDO FEATURE TABLE =====")

if "fe" not in globals() or "df_ratio" not in globals():
    raise RuntimeError(
        "Execute as celulas anteriores deste notebook primeiro."
    )

if not spark.catalog.tableExists(tabela_features):
    raise RuntimeError(
        "Feature Table ainda nao criada."
    )

tabela_atual = spark.table(tabela_features)

if tabela_atual.count() != 150:
    raise RuntimeError("Feature Table com contagem incorreta")

if "species" in tabela_atual.columns:
    raise RuntimeError("Target species presente nas features")

if tabela_atual.select("id_flor").distinct().count() != 150:
    raise RuntimeError("Chaves duplicadas")

fe.get_table(name=tabela_features)

print("FEATURE TABLE VALIDADA:", tabela_features)


## Adicionando `petal_ratio` com MERGE

O método `write_table` atualiza a Feature Table
utilizando `id_flor` como chave primária.

Se a coluna já estiver preenchida, não repetiremos
a gravação.


In [ ]:
print("===== ADICIONANDO PETAL_RATIO =====")

tabela_atual = spark.table(tabela_features)

if "petal_ratio" in tabela_atual.columns:
    nulos = tabela_atual.filter(
        tabela_atual["petal_ratio"].isNull()
    ).limit(1).count()
else:
    nulos = 1

if nulos == 0:
    print("PETAL_RATIO JA EXISTE: GRAVACAO DISPENSADA")
else:
    fe.write_table(
        name=tabela_features,
        df=df_ratio,
        mode="merge"
    )
    print("FEATURE ADICIONADA COM MERGE")

resultado = spark.table(tabela_features)

if "petal_ratio" not in resultado.columns:
    raise RuntimeError("Feature nao encontrada apos MERGE")

display(
    resultado.select(
        "id_flor", "petal_length",
        "petal_width", "petal_ratio"
    ).limit(10)
)


## Histórico Delta Lake

Consultar as alterações realizadas na Feature Table.

O Time Travel permite consultar versões preservadas,
mas o controle de vazamento temporal também exige
considerar a data de disponibilidade das features.


In [ ]:
print("===== HISTORICO DELTA =====")

display(
    spark.sql(
        f"DESCRIBE HISTORY {tabela_features}"
    )
)

print("HISTORICO CONSULTADO")


## Demonstração de DROP COLUMN

Esta demonstração é opcional.

Por segurança, a remoção ocorrerá somente em uma
tabela de laboratório, não na Feature Store principal.

Para executar, altere o widget `demonstrar_drop`
para `sim`.


In [ ]:
dbutils.widgets.dropdown(
    "demonstrar_drop",
    "nao",
    ["nao", "sim"]
)

if dbutils.widgets.get("demonstrar_drop") == "nao":
    print("DROP NAO EXECUTADO")

else:
    partes = tabela_features.rsplit(".", 1)
    tabela_lab = partes[0] + ".iris_features_drop_lab"

    if not spark.catalog.tableExists(tabela_lab):
        spark.sql(
            f"CREATE TABLE {tabela_lab} USING DELTA AS "
            f"SELECT id_flor, petal_ratio FROM {tabela_features}"
        )

    if "petal_ratio" in spark.table(tabela_lab).columns:
        spark.sql(
            f"ALTER TABLE {tabela_lab} "
            "SET TBLPROPERTIES "
            "('delta.columnMapping.mode' = 'name')"
        )

        spark.sql(
            f"ALTER TABLE {tabela_lab} "
            "DROP COLUMN petal_ratio"
        )

        print("COLUNA REMOVIDA SOMENTE DO LABORATORIO")
    else:
        print("COLUNA JA AUSENTE NO LABORATORIO")

    display(spark.table(tabela_lab).limit(5))
    print("FEATURE STORE PRINCIPAL PRESERVADA")


## Validação final

A tabela oficial deve continuar com 150 registros,
chaves únicas, sem o target e com `petal_ratio`.


In [ ]:
final = spark.table(tabela_features)

if final.count() != 150:
    raise RuntimeError("Contagem final invalida")

if "species" in final.columns:
    raise RuntimeError("Target presente indevidamente")

if "petal_ratio" not in final.columns:
    raise RuntimeError("Feature petal_ratio ausente")

if final.select("id_flor").distinct().count() != 150:
    raise RuntimeError("Chaves duplicadas")

print("===== FEATURE STORE VALIDADA =====")
print("TABELA:", tabela_features)
print("REGISTROS: 150")
print("COLUNAS:", final.columns)
print("TARGET EXCLUIDO")
print("PETAL_RATIO PRESENTE")
